# UN Sentiment & Tone Analysis

This analysis examines sentiment and tone in UN Noon Briefings by classifying briefing sections as Positive, Neutral, or Negative and identifying their communicative tone, such as Diplomatic, Concerned, Critical, Urgent, Optimistic, Neutral/Informational, or Constructive.

# 1. Set up and load data
This notebook uses the pre-uploaded database specified by `BRIEFINGS_DB_PATH`.

Alternatively, the database can be created from scratch for a desired date range:

```bash
briefings update 2026-09-01 2026-09-10

briefings init-db

briefings load 2026-09-01 2026-09-10
```

In [2]:
import pandas as pd
import ollama
from tqdm import tqdm
import json

from un_llm_analysis.config import BRIEFINGS_DB_PATH
from un_llm_analysis.database import (
    get_db_connection,
    get_briefings_between_dates,
    insert_into_analysis_run,
    insert_sentiment_analysis,
    get_analysis_run,
    get_sentiment_results_between_dates
    
)
from un_llm_analysis.analysis.sentimentv2 import (
    get_existing_sentiment_analysis,
    run_sentiment_analysis,
    
)


In [3]:
from un_llm_analysis.config import BRIEFINGS_DB_PATH

print("Database path:", BRIEFINGS_DB_PATH)

Database path: E:\correlaid-mentoring-project\app\briefings.db


In [4]:
model_name = "llama3.2:latest"
start_date = "2026-08-01"
end_date = "2026-08-10"

temperature = 0
overwrite = False

In [5]:
import importlib
import un_llm_analysis.analysis.sentimentv2 as sentiment_module

importlib.reload(sentiment_module)

<module 'un_llm_analysis.analysis.sentimentv2' from 'E:\\correlaid-mentoring-project\\un_llm_analysis\\analysis\\sentimentv2.py'>

In [6]:
# Get the data in the database with its maximum date range
#db_start_date = "2026-09-01"
#db_end_date = "2026-09-10"

with get_db_connection(BRIEFINGS_DB_PATH) as conn:
    briefings = get_briefings_between_dates(conn,start_date,end_date)

2026-11-04 07:07:19.298 | INFO     | un_llm_analysis.database:get_briefings_between_dates:317 - 8 briefings obtained between '2026-08-01' and '2026-08-10'


In [7]:
# Checking the content in the database
conn = get_db_connection()


cursor = conn.cursor()

tables = [
        "briefing",
        "section",
        "analysis_run",
        "sentiment_analysis",
    ]

for table in tables:
        cursor.execute(f"SELECT COUNT(*) FROM {table}")
        count = cursor.fetchone()[0]
        print(f"{table}: {count} records")



briefing: 25 records
section: 189 records
analysis_run: 2 records
sentiment_analysis: 141 records


In [8]:
# Get the section data from the briefings and store it in a DataFrame

rows = []

for briefing in tqdm(briefings):
    for i, section in enumerate(briefing.sections, start=1):
        rows.append(
            {
                "section_id": section.section_id,
                "briefing_id": section.briefing_id,
                "briefing_date": briefing.briefing_date,
                "section_title": section.title,
                "section_text": section.text,
            }
        )

section_data = pd.DataFrame(rows)

section_data.info()
section_data.tail(5)

100%|██████████| 8/8 [00:00<00:00, 2097.81it/s]


<class 'pandas.DataFrame'>
RangeIndex: 57 entries, 0 to 56
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   section_id     57 non-null     int64
 1   briefing_id    57 non-null     int64
 2   briefing_date  57 non-null     str  
 3   section_title  57 non-null     str  
 4   section_text   57 non-null     str  
dtypes: int64(2), str(3)
memory usage: 73.6 KB


,section_id,briefing_id,briefing_date,section_title,section_text
52,2026081005,20260810,2026-08-10,OCCUPIED PALESTINIAN TERRITORY,"On the Occupied Palestinian Territory, the Off..."
53,2026081006,20260810,2026-08-10,SUDAN,The Office for the Coordination of Humanitaria...
54,2026081007,20260810,2026-08-10,RWANDA,"In Rwanda, the UN and its humanitarian partner..."
55,2026081008,20260810,2026-08-10,NAGASAKI,"Yesterday, Izumi Nakamitsu, the High Represent..."
56,2026081009,20260810,2026-08-10,INFORMATION INTEGRITY,"Ahead of International Youth Day on 12 August,..."


# Install the required Python packages


`pipenv install mlflow openai`

# Open a second terminal in your project folder and run MLflow server

`pipenv run mlflow server --host 127.0.0.1 --port 5000`

In [ ]:
summary = run_sentiment_analysis(
    section_data=section_data,
    start_date=start_date,
    end_date=end_date,
    model_name=model_name,
    temperature=temperature,
    overwrite=overwrite,
)

print(summary)

No matching analysis found for 2026-08-01 to 2026-08-10 using model 'llama3.2:latest'.


Classifying with llama3.2:latest:  88%|████████▊ | 50/57 [09:38<01:10, 10.12s/it]

# Reference



1. This notebook is based on the following notebook.
[Sentiment Analysis Issue Mapping  Notebook](notebooks/franz/02_analyze_issue_mapping.ipynb).


In [5]:
import sqlite3
from un_llm_analysis.config import BRIEFINGS_DB_PATH


conn = sqlite3.connect(BRIEFINGS_DB_PATH)


print("Database path:", BRIEFINGS_DB_PATH)
tables = conn.execute("""
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
""").fetchall()

print(tables)

conn.close()

Database path: E:\correlaid-mentoring-project\app\briefings.db
[('briefing',), ('section',), ('sdg',), ('issue',), ('analysis_run',), ('sqlite_sequence',), ('sentiment_analysis',), ('country',)]
